# Phí ship Olist: hồi quy log-log, chẩn đoán residual và SE đúng

Đội logistics hỏi: phí ship phụ thuộc cân nặng, kích thước, khoảng cách thế nào? Bạn dựng bảng từ
6 tệp Olist, fit OLS dạng log-log, chẩn đoán residual, và chọn SE phù hợp với cấu trúc dữ liệu
(nhiều đơn của cùng một seller).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/05-econometrics-regression/exercises/intermediate/01-olist-freight-ols -q`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=["order_delivered_customer_date"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
products = pd.read_csv(RAW / "olist_products_dataset.csv")
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})
print(len(items), len(orders), len(geo))

## 1. Khoảng cách người bán – người mua

Bảng geolocation có nhiều toạ độ cho mỗi zip prefix (5 số đầu của CEP). Lấy **trung vị** lat/lng mỗi
prefix (ít nhạy với toạ độ sai hơn trung bình), rồi tính khoảng cách đường chim bay bằng haversine
với bán kính Trái Đất 6 371 km.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Khoảng cách đường tròn lớn (km), nhận số hoặc mảng, toạ độ theo độ. Bán kính 6 371 km."""
    # TODO: đổi sang radian, công thức haversine
    raise NotImplementedError

## 2. Dựng bảng phân tích

Quy tắc (viết trong docstring, test kiểm từng quy tắc):

- Đơn **giao xong**: `order_status == "delivered"` và có `order_delivered_customer_date`.
- Chỉ đơn có **đúng một món**, để phí ship gắn rõ với một sản phẩm.
- Bỏ dòng thiếu toạ độ, cân nặng, kích thước; bỏ `freight_value <= 0`, cân nặng `<= 0`, khoảng cách
  `<= 0` (cùng zip prefix: không đo được).

In [ ]:
def build_freight_table(items, orders, customers, sellers, products, geo) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có đúng một món. Cột zip prefix đọc dạng str.

    Cột trả về: order_id, seller_id, freight_value, price, product_weight_g, volume_cm3, distance_km,
    same_state (0/1: bang khách == bang seller), log_freight, log_weight, log_volume,
    log_distance (= log1p(distance_km)), log_price.
    """
    # TODO: lọc đơn giao xong và đơn đúng một món
    # TODO: nối customers (qua orders.customer_id), sellers, products
    # TODO: trung vị lat/lng theo geolocation_zip_code_prefix, nối cho khách và seller, haversine
    # TODO: volume_cm3 = dài * cao * rộng; same_state; lọc dòng thiếu / <= 0; các cột log
    raise NotImplementedError

In [ ]:
# explore: bảng phân tích trên dữ liệu thật
d = build_freight_table(items, orders, customers, sellers, products, geo)
print(len(d), d.seller_id.nunique())
d.describe().T

## 3. Fit model log-log với SE phù hợp

`FORMULA` cố định để mọi người so được kết quả. `fit_freight` nhận `cov_type`:
- `"nonrobust"`: SE cổ điển.
- `"HC3"`: robust với heteroskedasticity.
- `"cluster"`: cluster theo `seller_id` (nhiều đơn cùng seller có sai số tương quan).

In [ ]:
FORMULA = "log_freight ~ log_weight + log_volume + log_distance + same_state + log_price"


def fit_freight(df: pd.DataFrame, cov_type: str = "nonrobust"):
    """OLS theo FORMULA; trả kết quả statsmodels. cov_type: nonrobust, HC3, cluster (theo seller_id)."""
    # TODO: smf.ols(FORMULA, df).fit(...); cluster cần cov_kwds={"groups": mã số nguyên của seller_id}
    raise NotImplementedError


def compare_se(df: pd.DataFrame) -> pd.DataFrame:
    """SE của mọi hệ số theo ba cách: cột classic, HC3, cluster; index là tên hệ số."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: kết quả và so sánh SE
res = fit_freight(d)
print(res.summary().tables[1])
se = compare_se(d)
se.assign(hc3_ratio=se.HC3 / se.classic, cluster_ratio=se.cluster / se.classic).round(4)

## 4. Chẩn đoán residual

Ba hình: residual vs fitted (dạng hàm, heteroskedasticity), QQ của residual (đuôi), và phân phối
residual trung bình theo seller (vì sao cần cluster). Sau đó thử bỏ `same_state` để thấy omitted
variable bias trên hệ số khoảng cách.

In [ ]:
# explore: residual vs fitted và QQ-plot
import matplotlib.pyplot as plt
import statsmodels.api as sm

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sample = np.random.default_rng(0).choice(len(d), 5000, replace=False)
ax[0].scatter(res.fittedvalues.iloc[sample], res.resid.iloc[sample], s=3, alpha=0.3)
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set(xlabel="fitted log_freight", ylabel="residual", title="Residual vs fitted (5 000 điểm)")
sm.qqplot(res.resid.iloc[sample], line="s", ax=ax[1])
ax[1].set_title("QQ-plot residual")
plt.tight_layout()
plt.show()

In [ ]:
# explore: residual trung bình theo seller (seller có ít nhất 30 đơn) và omitted variable bias
from statsmodels.stats.diagnostic import het_breuschpagan

by_seller = res.resid.groupby(d.seller_id).agg(["mean", "size"]).query("size >= 30")
print("độ lệch chuẩn residual trung bình giữa các seller:", round(by_seller["mean"].std(), 3))
print("Breusch-Pagan LM:", round(het_breuschpagan(res.resid, res.model.exog)[0]))
no_state = smf.ols("log_freight ~ log_weight + log_volume + log_distance + log_price", d).fit()
print("log_distance có / không có same_state:", round(res.params.log_distance, 4), round(no_state.params.log_distance, 4))

## 5. Diễn giải cho đội logistics

`effects(res)` quy hệ số ra phần trăm thay đổi phí ship:
- `distance_doubling`: khoảng cách gấp đôi, $$2^{\beta} - 1$$ (xấp xỉ, vì model dùng `log1p`).
- `same_state`: cùng bang so với khác bang, $$e^{\beta} - 1$$.
- `weight_10pct`: nặng hơn 10%, $$1.1^{\beta} - 1$$.

In [ ]:
def effects(res) -> dict[str, float]:
    """Phần trăm thay đổi phí ship (0.10 nghĩa là +10%): distance_doubling, same_state, weight_10pct."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore
effects(res)

## 6. Viết kết luận

Viết 5–8 câu cho đội logistics: các yếu tố chính và độ lớn, SE nào bạn báo cáo và vì sao, model này
**không** nói được gì (hệ số `log_price`, nhân quả), và bước tiếp theo bạn đề xuất.